In [40]:
import numpy as np
import pandas as pd
import datetime
import os

In [41]:
# Replace 'your_directory_path' with the path to your folder
folder_path = 'Runner_final'

# List all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

runners_final_df = pd.DataFrame()

for file in csv_files:
    df = pd.read_csv(f'Runner_final/{file}')
        
    runners_final_df = pd.concat([runners_final_df, df])
    

In [42]:
# Replace 'your_directory_path' with the path to your folder
folder_path = 'catchers'

# List all CSV files in the folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

catchers_final_df = pd.DataFrame()

for file in csv_files:
    df = pd.read_csv(f'catchers/{file}')
        
    catchers_final_df = pd.concat([catchers_final_df, df])
    

In [43]:
catchers_final_df.replace(' --', np.nan, inplace=True)

In [44]:
catchers_final_df['Arm_Strength'] = catchers_final_df['Arm_Strength'].astype(float)

catchers_final_df['Exchange'] = catchers_final_df['Exchange'].astype(float)

In [45]:
runners_final_df.replace(' --', np.nan, inplace=True)

In [46]:
stealing_df = pd.read_csv('combined.csv')

stealing_df.drop(columns=['Unnamed: 0'], inplace=True)

stealing_df.head()

,Start_Base,End_Base,Event,Respon_Pitcher,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,batter_name,pitch_hand,credit,credit_id,homeScore,awayScore,inning,isTopInning,Date,Action,Base
0,1B,NaN,Caught Stealing 2B,NaN,545341,Randal Grichuk,643338,Chad Green,543760,Marcus Semien,R,Gary Sánchez,596142,2,2,7,True,2021-04-01,Caught,2B
1,1B,2B,Stolen Base 2B,NaN,643565,Mike Tauchman,605447,Jordan Romano,457803,Jay Bruce,R,Danny Jansen,643376,2,2,9,False,2021-04-01,Stolen,2B
2,2B,3B,Stolen Base 3B,NaN,643565,Mike Tauchman,605447,Jordan Romano,640449,Clint Frazier,R,Danny Jansen,643376,2,2,9,False,2021-04-01,Stolen,3B
3,1B,2B,Stolen Base 2B,NaN,666185,Dylan Carlson,489334,Craig Stammen,425877,Yadier Molina,R,Austin Nola,543592,3,6,4,True,2020-09-30,Stolen,2B
4,2B,3B,Stolen Base 3B,NaN,502054,Tommy Pham,650893,Génesis Cabrera,543592,Austin Nola,L,Yadier Molina,425877,4,6,6,False,2020-09-30,Stolen,3B


In [47]:
stealing_df['Result'] = stealing_df['Action'].apply(lambda x : 1 if x =='Stolen' else 0)

In [81]:
data_dict = []

final_df = {'Result':[], 'Exchange':[] ,'Arm_Strength':[], 'Sprint':[]}

for row, j in stealing_df.iterrows():
    catcher_id = j['credit_id']
    
    runner_id = j['pitcher_id']
    
    catcher_name = j['credit']
    
    runner_name = j['runner_name']
    
    date = j['Date']
    
    if len(runners_final_df.loc[(runners_final_df['runner_name'] == runner_name) & (runners_final_df['Date'] == date)]) == 0:
        continue
    
    runner_sprint = runners_final_df.loc[(runners_final_df['runner_name'] == runner_name) & (runners_final_df['Date'] == date)]['Sprint'].tolist()[0]
    
    try:

        catcher_df = pd.read_csv(f'catchers/{catcher_name}.csv')
        
        
    except:
        print(catcher_name)
        continue
        
    if 'Exchange' not in catcher_df.columns:
        continue
        
    catcher_df['Result'] = catcher_df['Action'].apply(lambda x : 1 if x =='Stolen' else 0)
        
    catcher_df.drop(columns=['Unnamed: 0'], inplace=True)

    catcher_df.drop_duplicates(inplace=True)

    catcher_past = catcher_df.loc[catcher_df['Date'] < date].sort_values(by='Date').tail(5)

    if len(catcher_past) < 5:
        continue

    catcher_past['Result'] = catcher_past['Action'].apply(lambda x : 1 if x =='Stolen' else 0)
    
    uniques = catcher_past['Arm_Strength'].unique()




    if catcher_past['Arm_Strength'].dtypes == 'object':
    
        arm = catcher_past['Arm_Strength'].tolist()
        numbers = [item.strip() for item in arm if item != ' --']

        if len(numbers) == 0:
            catcher_past['Arm_Strength'].replace(' --', np.nan, inplace=True)
            catcher_past['Arm_Strength'] = catcher_past['Arm_Strength'].astype(float)
            
        else:
            
            repl = float(numbers[0])
            catcher_past['Arm_Strength'].replace(' --', repl, inplace=True)
            catcher_past['Arm_Strength'] = catcher_past['Arm_Strength'].astype(float)

        
    if catcher_past['Exchange'].dtypes == 'object':
    
        arm = catcher_past['Exchange'].tolist()
        numbers = [item.strip() for item in arm if item != ' --']
        if len(numbers) == 0:
            catcher_past['Exchange'].replace(' --', np.nan, inplace=True)
            catcher_past['Exchange'] = catcher_past['Exchange'].astype(float)
            
        else:
            
            repl = float(numbers[0])
            catcher_past['Exchange'].replace(' --', repl, inplace=True)
            catcher_past['Exchange'] = catcher_past['Exchange'].astype(float)

    sprints = []

    for row, i in catcher_past.iterrows():

        runner = i['runner_name']

        current_year = i['Year'] 
        
        if len(runners_final_df.loc[(runners_final_df['runner_name'] == runner) & (runners_final_df['Current_Year'] == current_year)]['Sprint']) == 0:
            sprints.append(np.nan)
            continue

        sprint = runners_final_df.loc[(runners_final_df['runner_name'] == runner) & (runners_final_df['Current_Year'] == current_year)]['Sprint'].tolist()[0]

        sprints.append(sprint)
        
    catcher_past['Sprint'] = sprints
    
    if catcher_past['Sprint'].dtypes == 'object':
    
        arm = catcher_past['Sprint'].tolist()
        numbers = [item.strip() for item in arm if item != np.nan]
        if len(numbers) == 0:
            catcher_past['Sprint'].replace(' --', np.nan, inplace=True)
            catcher_past['Sprint'] = catcher_past['Sprint'].astype(float)
            
        else:
            
            repl = float(numbers[0])
            catcher_past['Sprint'].replace(np.nan, repl, inplace=True)
            catcher_past['Sprint'] = catcher_past['Sprint'].astype(float)

    past_data = catcher_past[['Result', 'Exchange' ,'Arm_Strength', 'Sprint', 'Date']].sort_values(by='Date', ascending=False)

    lag = 1

    cols = ['Result', 'Exchange' ,'Arm_Strength', 'Sprint']

    for row, i in past_data.iterrows():

        for col in cols:

            if f'{col}_{lag}' not in final_df:
                final_df[f'{col}_{lag}'] = []

            final_df[f'{col}_{lag}'].append(i[col])

        lag += 1
    
    Exchange = catcher_df.loc[catcher_df['Date'] < date]['Exchange'].tolist()[0]
    
    Result = catcher_df.loc[catcher_df['Date'] < date]['Result'].tolist()[0]
    
    Arm_Strength = catcher_df.loc[catcher_df['Date'] < date]['Arm_Strength'].tolist()[0]
    
    final_df['Result'].append(Result)
    final_df['Exchange'].append(Exchange)
    final_df['Arm_Strength'].append(Arm_Strength)
    final_df['Sprint'].append(runner_sprint)
    

In [82]:
stolen_df = pd.DataFrame(final_df)
stolen_df.to_csv('stolen_df_2.csv', index=False)

In [83]:
stolen_df.isna().sum()

Result               0
Exchange             0
Arm_Strength         0
Sprint               0
Result_1             0
Exchange_1         435
Arm_Strength_1    1361
Sprint_1          8758
Result_2             0
Exchange_2         435
Arm_Strength_2    1361
Sprint_2          8498
Result_3             0
Exchange_3         435
Arm_Strength_3    1361
Sprint_3          8269
Result_4             0
Exchange_4         435
Arm_Strength_4    1361
Sprint_4          8190
Result_5             0
Exchange_5         435
Arm_Strength_5    1361
Sprint_5          8333
dtype: int64

In [73]:
stolen_df

,Result,Exchange,Arm_Strength,Sprint,Result_1,Exchange_1,Arm_Strength_1,Sprint_1,Result_2,Exchange_2,...,Arm_Strength_8,Sprint_8,Result_9,Exchange_9,Arm_Strength_9,Sprint_9,Result_10,Exchange_10,Arm_Strength_10,Sprint_10
0,1,0.74,84.9,28.0,1,0.74,84.9,29.2,1,0.74,...,84.9,25.9,1,0.74,84.9,25.0,1,0.74,84.9,27.6
1,0,0.75,81.4,27.3,0,0.75,81.4,28.7,1,0.75,...,81.4,28.2,1,0.75,81.4,25.7,1,0.75,81.4,28.1
2,0,0.75,81.4,27.3,0,0.75,81.4,28.7,1,0.75,...,81.4,28.2,1,0.75,81.4,25.7,1,0.75,81.4,28.1
3,0,0.73,79.0,27.9,0,0.73,79.0,27.9,1,0.73,...,79.0,26.5,0,0.73,79.0,NaN,0,0.73,79.0,26.9
4,1,0.7,83.2,27.8,1,0.70,83.2,29.4,0,0.70,...,83.2,27.4,1,0.70,83.2,29.4,1,0.70,83.2,29.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35814,1,0.66,82.6,27.5,1,0.66,82.6,28.2,1,0.66,...,82.6,27.7,1,0.66,82.6,26.4,0,0.66,82.6,26.2
35815,0,0.75,80.1,27.9,0,0.75,80.1,29.1,1,0.75,...,80.1,28.4,1,0.75,80.1,28.4,0,0.75,80.1,28.8
35816,1,0.75,80.1,28.8,1,0.75,80.1,28.9,0,0.75,...,80.1,28.4,1,0.75,80.1,28.4,0,0.75,80.1,28.8
35817,1,0.75,80.1,28.8,1,0.75,80.1,28.9,1,0.75,...,80.1,28.8,0,0.75,80.1,26.3,0,0.75,80.1,26.3


In [72]:
stolen_df.dropna()

,Result,Exchange,Arm_Strength,Sprint,Result_1,Exchange_1,Arm_Strength_1,Sprint_1,Result_2,Exchange_2,...,Arm_Strength_8,Sprint_8,Result_9,Exchange_9,Arm_Strength_9,Sprint_9,Result_10,Exchange_10,Arm_Strength_10,Sprint_10
0,1,0.74,84.9,28.0,1,0.74,84.9,29.2,1,0.74,...,84.9,25.9,1,0.74,84.9,25.0,1,0.74,84.9,27.6
18,0,0.72,76.7,25.9,0,0.72,76.7,28.7,1,0.72,...,76.7,27.0,1,0.72,76.7,26.1,1,0.72,76.7,29.4
19,0,0.72,76.7,25.0,0,0.72,76.7,28.7,1,0.72,...,76.7,27.0,1,0.72,76.7,26.1,1,0.72,76.7,29.4
23,1,0.64,74.2,27.4,1,0.64,74.2,29.1,1,0.64,...,74.2,28.5,1,0.64,74.2,28.1,1,0.64,74.2,25.9
27,1,0.67,76.5,29.7,1,0.67,76.5,28.6,1,0.67,...,76.5,28.7,1,0.67,76.5,25.5,1,0.67,76.5,29.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35779,1,0.79,82.0,27.1,1,0.79,82.0,28.0,0,0.79,...,82.0,28.2,1,0.79,82.0,29.3,1,0.79,82.0,25.5
35793,1,0.76,77.9,27.7,1,0.76,77.9,26.0,1,0.76,...,77.9,27.3,1,0.76,77.9,25.9,1,0.76,77.9,28.9
35794,1,0.79,82.0,27.7,1,0.79,82.0,27.4,1,0.79,...,82.0,27.4,1,0.79,82.0,25.5,1,0.79,82.0,27.7
35795,1,0.79,82.0,28.0,1,0.79,82.0,27.4,1,0.79,...,82.0,27.4,1,0.79,82.0,25.5,1,0.79,82.0,27.7


In [69]:
len(stolen_df)

35819

In [30]:
final_df

,Result_1,Exchange_1,Arm_Strength_1,Sprint_1,Result_2,Exchange_2,Arm_Strength_2,Sprint_2,Result_3,Exchange_3,...,Sprint_28,Result_29,Exchange_29,Arm_Strength_29,Sprint_29,Result_30,Exchange_30,Arm_Strength_30,Sprint_30,Sprint_31


In [11]:
catcher_past

,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,batter_name,pitch_hand,...,inning,isTopInning,Date,Action,Base,Year,Current_Year,Arm_Strength,Exchange,Result
51,1B,NaN,Caught Stealing 2B,543760,Marcus Semien,502624,Chase Anderson,543257,Robbie Grossman,R,...,1,False,2019-08-01,Caught,2B,2019,2020,77.5,0.67,0
53,1B,2B,Stolen Base 2B,592192,Mark Canha,543351,Jay Jackson,595777,Jurickson Profar,R,...,8,False,2019-08-01,Stolen,2B,2019,2020,77.5,0.67,1
52,2B,3B,Stolen Base 3B,592192,Mark Canha,502624,Chase Anderson,595777,Jurickson Profar,R,...,2,False,2019-08-01,Stolen,3B,2019,2020,77.5,0.67,1
50,1B,NaN,Caught Stealing 2B,462101,Elvis Andrus,519141,Drew Pomeranz,596059,Rougned Odor,L,...,8,True,2019-08-10,Caught,2B,2019,2020,77.5,0.67,0
49,2B,3B,Stolen Base 3B,665742,Juan Soto,448855,Junior Guerra,605452,Joe Ross,R,...,14,False,2019-08-17,Stolen,3B,2019,2020,77.5,0.67,1
48,2B,3B,Stolen Base 3B,622168,Yairo Muñoz,605288,Adrian Houser,543939,Kolten Wong,R,...,5,False,2019-08-21,Stolen,3B,2019,2020,77.5,0.67,1
47,1B,2B,Stolen Base 2B,543939,Kolten Wong,642207,Devin Williams,657041,Lane Thomas,R,...,9,True,2019-08-27,Stolen,2B,2019,2020,77.5,0.67,1
46,1B,NaN,Caught Stealing 2B,451594,Dexter Fowler,605288,Adrian Houser,502671,Paul Goldschmidt,R,...,1,True,2019-08-27,Caught,2B,2019,2020,77.5,0.67,0
45,1B,2B,Stolen Base 2B,502671,Paul Goldschmidt,543475,Jordan Lyles,542303,Marcell Ozuna,R,...,6,True,2019-08-28,Stolen,2B,2019,2020,77.5,0.67,1
44,1B,2B,Stolen Base 2B,647351,Abraham Toro,605288,Adrian Houser,455117,Martín Maldonado,R,...,2,True,2019-09-02,Stolen,2B,2019,2020,77.5,0.67,1


In [213]:
pd.DataFrame(data_dict)

,Sprint_runner,Exchange_runner,Result_runner,Arm_Strength_runner,Result_catcher,Exchange_catcher,Arm_Strength_catcher,Sprint_catcher,Result
0,28.013333,0.728000,0.633333,79.724000,0.733333,0.732,86.480000,27.455172,0
1,27.260000,0.712222,0.900000,80.222222,0.766667,0.750,81.400000,28.073077,1
2,27.260000,0.712222,0.900000,80.222222,0.766667,0.750,81.400000,28.073077,1
3,27.960000,0.720000,1.000000,77.214286,0.666667,0.706,82.300000,28.318750,1
4,29.900000,0.732500,0.900000,82.175000,0.633333,0.750,84.376667,27.542857,1
...,...,...,...,...,...,...,...,...,...
22791,30.000000,NaN,0.600000,NaN,0.600000,0.750,80.100000,27.986957,0
22792,30.000000,NaN,0.600000,NaN,0.633333,0.750,80.100000,28.004167,0
22793,28.180000,NaN,0.900000,NaN,0.700000,0.750,80.100000,28.083333,0
22794,27.340000,NaN,0.866667,NaN,0.700000,0.750,80.100000,28.083333,0


In [206]:
catcher_past['Arm_Strength'].unique()

array([' --'], dtype=object)

In [ ]:
uniques = catcher_past['Arm_Strength'].unique()

if (len(uniques) == 1):
    catcher_past['Arm_Strength'].replace(' --', np.nan)
    catcher_past['Arm_Strength'] = catcher_past['Arm_Strength'].astype(float)
    

In [155]:
catcher_past

,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,batter_name,pitch_hand,...,inning,isTopInning,Date,Action,Base,Year,Current_Year,Arm_Strength,Exchange,Result
28,1B,NaN,Caught Stealing 2B,608577,Nomar Mazara,605135,Chris Bassitt,596059,Rougned Odor,R,...,6,False,2019-06-08,Caught,2B,2019,2020,--,0.71,0
31,1B,2B,Stolen Base 2B,452254,Hunter Pence,621112,Paul Blackburn,452678,Asdrúbal Cabrera,R,...,1,False,2019-06-08,Stolen,2B,2019,2020,--,0.71,1
27,3B,score,Stolen Base Home,596059,Rougned Odor,488748,Ryan Buchter,425783,Shin-Soo Choo,L,...,8,False,2019-06-09,Stolen,Home,2019,2020,--,0.71,1
26,1B,2B,Stolen Base 2B,595281,Kevin Kiermaier,571666,Mike Fiers,572287,Mike Zunino,R,...,4,False,2019-06-11,Stolen,2B,2019,2020,--,0.71,1
25,3B,score,Stolen Base Home,642715,Willy Adames,571666,Mike Fiers,572287,Mike Zunino,R,...,4,False,2019-06-11,Stolen,Home,2019,2020,--,0.71,1
24,1B,2B,Stolen Base 2B,543829,Dee Strange-Gordon,623913,Wei-Chung Wang,605480,Mallex Smith,L,...,6,True,2019-06-14,Stolen,2B,2019,2020,--,0.71,1
23,1B,NaN,Caught Stealing 2B,642082,Chance Sisco,571666,Mike Fiers,600474,Pedro Severino,R,...,1,True,2019-06-17,Caught,2B,2019,2020,--,0.71,0
22,1B,2B,Stolen Base 2B,592859,Stevie Wilkerson,433589,Yusmeiro Petit,542979,Keon Broxton,R,...,7,True,2019-06-19,Stolen,2B,2019,2020,--,0.71,1
21,1B,2B,Stolen Base 2B,541645,Avisaíl García,595014,Blake Treinen,596847,Ji Man Choi,R,...,9,True,2019-06-20,Stolen,2B,2019,2020,--,0.71,1
20,1B,2B,Stolen Base 2B,664040,Brandon Lowe,595014,Blake Treinen,541645,Avisaíl García,R,...,9,True,2019-06-20,Stolen,2B,2019,2020,--,0.71,1


In [145]:
mean_catch

Result           0.833333
Arm_Strength    82.750000
Sprint                NaN
dtype: float64

In [140]:
runner_past

,Unnamed: 0.1,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,batter_name,...,isTopInning,Date,Action,Base,Year,Current_Year,Sprint,Result,Exchange,Arm_Strength
40,40,1B,2B,Stolen Base 2B,545341,Randal Grichuk,542432,José Ramirez,649557,Aledmys Díaz,...,True,2017-05-06,Stolen,2B,2017,2018,28.3,1,0.75,--
39,39,1B,2B,Stolen Base 2B,545341,Randal Grichuk,527054,Julio Teheran,649557,Aledmys Díaz,...,True,2017-05-06,Stolen,2B,2017,2018,28.3,1,0.75,--
46,46,1B,2B,Stolen Base 2B,545341,Randal Grichuk,527054,Julio Teheran,649557,Aledmys Díaz,...,True,2017-05-06,Stolen,2B,2017,2018,28.3,1,0.75,--
45,45,2B,NaN,Caught Stealing 3B,545341,Randal Grichuk,285079,R.A. Dickey,576397,Jedd Gyorko,...,True,2017-05-07,Caught,3B,2017,2018,28.3,0,0.75,--
38,38,2B,NaN,Caught Stealing 3B,545341,Randal Grichuk,285079,R.A. Dickey,576397,Jedd Gyorko,...,True,2017-05-07,Caught,3B,2017,2018,28.3,0,0.75,--
44,44,1B,2B,Stolen Base 2B,545341,Randal Grichuk,451584,Wade Davis,543939,Kolten Wong,...,False,2017-05-12,Stolen,2B,2017,2018,28.3,1,0.75,--
37,37,1B,2B,Stolen Base 2B,545341,Randal Grichuk,451584,Wade Davis,543939,Kolten Wong,...,False,2017-05-12,Stolen,2B,2017,2018,28.3,1,0.75,--
43,43,1B,2B,Stolen Base 2B,545341,Randal Grichuk,519043,Matt Moore,543939,Kolten Wong,...,False,2017-05-19,Stolen,2B,2017,2018,28.3,1,0.75,--
36,36,1B,2B,Stolen Base 2B,545341,Randal Grichuk,519043,Matt Moore,543939,Kolten Wong,...,False,2017-05-19,Stolen,2B,2017,2018,28.3,1,0.75,--
42,42,1B,2B,Stolen Base 2B,545341,Randal Grichuk,622766,Miguel Díaz,500874,José A. Martínez,...,True,2017-09-05,Stolen,2B,2017,2018,28.3,1,0.75,--


In [137]:
catchers_final_df.loc[(catchers_final_df['credit'] == catcher)]



,Unnamed: 0,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,batter_name,...,awayScore,inning,isTopInning,Date,Action,Base,Year,Current_Year,Arm_Strength,Exchange


In [126]:
catcher_df.loc[(catcher_df['credit'] == catcher) & (catcher_df['Current_Year'] == current_year)]

,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,batter_name,pitch_hand,...,awayScore,inning,isTopInning,Date,Action,Base,Year,Current_Year,Arm_Strength,Exchange


In [125]:
runner_past

,Unnamed: 0.1,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,batter_name,...,isTopInning,Date,Action,Base,Year,Current_Year,Sprint,Result,Exchange,Arm_Strength
30,30,1B,2B,Stolen Base 2B,663656,Kyle Tucker,579328,Yusei Kikuchi,545350,Jake Marisnick,...,True,2019-09-25,Stolen,2B,2019,2020,27.6,1,0.75,--
35,35,1B,2B,Stolen Base 2B,663656,Kyle Tucker,579328,Yusei Kikuchi,545350,Jake Marisnick,...,True,2019-09-25,Stolen,2B,2019,2020,27.6,1,0.75,--
40,40,1B,2B,Stolen Base 2B,663656,Kyle Tucker,579328,Yusei Kikuchi,545350,Jake Marisnick,...,True,2019-09-25,Stolen,2B,2019,2020,27.6,1,0.75,--
19,19,1B,2B,Stolen Base 2B,663656,Kyle Tucker,608665,Kendall Graveman,572863,Dustin Garneau,...,False,2020-07-27,Stolen,2B,2020,2021,27.4,1,0.75,--
9,9,1B,2B,Stolen Base 2B,663656,Kyle Tucker,608665,Kendall Graveman,572863,Dustin Garneau,...,False,2020-07-27,Stolen,2B,2020,2021,27.4,1,0.75,--
29,29,1B,2B,Stolen Base 2B,663656,Kyle Tucker,608665,Kendall Graveman,572863,Dustin Garneau,...,False,2020-07-27,Stolen,2B,2020,2021,27.4,1,0.75,--
18,18,1B,2B,Stolen Base 2B,663656,Kyle Tucker,592135,Cam Bedrosian,455117,Martín Maldonado,...,True,2020-07-31,Stolen,2B,2020,2021,27.4,1,0.75,--
8,8,1B,2B,Stolen Base 2B,663656,Kyle Tucker,592135,Cam Bedrosian,455117,Martín Maldonado,...,True,2020-07-31,Stolen,2B,2020,2021,27.4,1,0.75,--
28,28,1B,2B,Stolen Base 2B,663656,Kyle Tucker,592135,Cam Bedrosian,455117,Martín Maldonado,...,True,2020-07-31,Stolen,2B,2020,2021,27.4,1,0.75,--
17,17,1B,2B,Stolen Base 2B,663656,Kyle Tucker,657277,Logan Webb,455117,Martín Maldonado,...,False,2020-08-10,Stolen,2B,2020,2021,27.4,1,0.75,--


In [124]:
numbers

['', '-', '-']

In [117]:
catcher_past['Arm_Strength'].dtypes == 'object'

True

In [112]:
catcher_past.dtypes

Start_Base       object
End_Base         object
Event            object
runner_id         int64
runner_name      object
pitcher_id        int64
pitcher_name     object
batter_id         int64
batter_name      object
pitch_hand       object
credit           object
credit_id         int64
homeScore         int64
awayScore         int64
inning            int64
isTopInning        bool
Date             object
Action           object
Base             object
Year              int64
Current_Year      int64
Arm_Strength    float64
Exchange        float64
Result            int64
dtype: object

In [93]:
catcher_past.dtypes

Start_Base       object
End_Base         object
Event            object
runner_id         int64
runner_name      object
pitcher_id        int64
pitcher_name     object
batter_id         int64
batter_name      object
pitch_hand       object
credit           object
credit_id         int64
homeScore         int64
awayScore         int64
inning            int64
isTopInning        bool
Date             object
Action           object
Base             object
Year              int64
Current_Year      int64
Arm_Strength     object
Exchange        float64
Result            int64
Sprint          float64
dtype: object

In [84]:
mean_catch

Result       0.766667
Exchange     0.674667
Sprint      27.780769
dtype: float64

In [80]:
i

Unnamed: 0.1                40
Start_Base                  1B
End_Base                    2B
Event           Stolen Base 2B
runner_id               545341
runner_name     Randal Grichuk
pitcher_id              542432
pitcher_name      José Ramirez
batter_id               649557
batter_name       Aledmys Díaz
pitch_hand                   R
credit           Tyler Flowers
credit_id               452095
homeScore                    0
awayScore                    5
inning                       7
isTopInning               True
Date                2017-05-06
Action                  Stolen
Base                        2B
Year                      2017
Current_Year              2018
Sprint                    28.3
Result                       1
Name: 40, dtype: object

In [74]:
i

Start_Base                      2B
End_Base                       NaN
Event           Caught Stealing 3B
runner_id                   466988
runner_name       Emilio Bonifácio
pitcher_id                  502154
pitcher_name          Zack Britton
batter_id                   645302
batter_name          Victor Robles
pitch_hand                       L
credit                Gary Sánchez
credit_id                   596142
homeScore                        2
awayScore                        3
inning                           9
isTopInning                  False
Date                    2020-07-26
Action                      Caught
Base                            3B
Year                          2020
Current_Year                  2021
Arm_Strength                  84.9
Exchange                      0.74
Result                           0
Name: 150, dtype: object

In [71]:
runners_df.loc[(runners_df['runner_name'] == runner)]

,Unnamed: 0.1,Unnamed: 0,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,...,homeScore,awayScore,inning,isTopInning,Date,Action,Base,Year,Current_Year,Sprint
0,0,4205,1B,2B,Stolen Base 2B,466988,Emilio Bonifácio,573109,AJ Ramos,611177,...,2,3,9,True,2016-09-23,Stolen,2B,2016,2017,28.7
1,1,170,1B,2B,Stolen Base 2B,466988,Emilio Bonifácio,573109,AJ Ramos,611177,...,2,3,9,True,2016-09-23,Stolen,2B,2016,2017,28.7
2,2,3577,1B,2B,Stolen Base 2B,466988,Emilio Bonifácio,573109,AJ Ramos,611177,...,2,3,9,True,2016-09-23,Stolen,2B,2016,2017,28.7
3,3,214,1B,2B,Stolen Base 2B,466988,Emilio Bonifácio,444468,Héctor Rondón,594809,...,0,0,8,True,2015-07-10,Stolen,2B,2015,2016,28.8
4,4,431,1B,NaN,Caught Stealing 2B,466988,Emilio Bonifácio,543935,Alex Wilson,594809,...,0,4,7,True,2015-06-28,Caught,2B,2015,2016,28.8
5,5,1444,1B,NaN,Caught Stealing 2B,466988,Emilio Bonifácio,543881,Pedro Villarreal,594809,...,5,2,6,False,2015-05-09,Caught,2B,2015,2016,28.8
6,6,1457,1B,NaN,Caught Stealing 2B,466988,Emilio Bonifácio,543881,Pedro Villarreal,594809,...,5,2,6,False,2015-05-08,Caught,2B,2015,2016,28.8


In [68]:
runner

'Emilio Bonifácio'

In [66]:
runners_df

,Unnamed: 0.1,Unnamed: 0,Start_Base,End_Base,Event,runner_id,runner_name,pitcher_id,pitcher_name,batter_id,...,homeScore,awayScore,inning,isTopInning,Date,Action,Base,Year,Current_Year,Sprint
0,0,2601,1B,2B,Stolen Base 2B,458675,Colby Rasmus,501925,Joe Smith,605227,...,6,6,8,True,2017-06-14,Stolen,2B,2017,2018,25.9
1,1,1973,1B,2B,Stolen Base 2B,458675,Colby Rasmus,501925,Joe Smith,605227,...,6,6,8,True,2017-06-14,Stolen,2B,2017,2018,25.9
2,2,6236,1B,2B,Stolen Base 2B,458675,Colby Rasmus,501817,Tony Barnette,472528,...,4,5,7,True,2016-06-06,Stolen,2B,2016,2017,27.2
3,3,6288,1B,2B,Stolen Base 2B,458675,Colby Rasmus,534910,Jesse Hahn,594828,...,2,0,1,False,2016-06-03,Stolen,2B,2016,2017,27.2
4,4,6692,1B,2B,Stolen Base 2B,458675,Colby Rasmus,518886,Craig Kimbrel,472528,...,6,7,9,True,2016-05-13,Stolen,2B,2016,2017,27.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,176,5425,1B,2B,Stolen Base 2B,606299,José Peraza,502032,Bud Norris,458015,...,4,3,5,True,2016-06-15,Stolen,2B,2016,2017,28.8
177,177,6108,1B,2B,Stolen Base 2B,606299,José Peraza,592791,Jameson Taillon,454975,...,0,4,1,False,2016-05-10,Stolen,2B,2016,2017,28.8
0,0,2003,1B,2B,Stolen Base 2B,670623,Isaac Paredes,605400,Aaron Nola,664040,...,0,1,4,False,2023-07-04,Stolen,2B,2023,2024,25.9
1,1,6417,1B,2B,Stolen Base 2B,670623,Isaac Paredes,605400,Aaron Nola,664040,...,0,1,4,False,2023-07-04,Stolen,2B,2023,2024,25.9


In [45]:
catcher_df.drop(columns=['Unnamed: 0'], inplace=True)

catcher_df.drop_duplicates(inplace=True)

catcher_past = catcher_df.loc[catcher_df['Date'] <= date].sort_values(by='Date').tail(30)

catcher_past['Result'] = catcher_past['Action'].apply(lambda x : 1 if x =='Stolen' else 0)

sprints = []

for row, i in catcher_past.iterrows():
    
    runner = i['runner_name']
    
    current_year = i['Current_Year']

    sprint = runners_df.loc[(runners_df['runner_name'] == 'Billy Hamilton') & (runners_df['Current_Year'] == 2021)]['Sprint'].tolist()[0]
    
    sprints.append(sprint)
catcher_past['Sprint'] = sprints

mean_catch = catcher_past[['Result', 'Exchange' ,'Arm_Strength', 'Sprint']].mean()

Result = mean_catch['Result']

Exchange = mean_catch['Exchange']

Arm_Strength = mean_catch['Arm_Strength']

Sprint = mean_catch['Sprint']

KeyError: "['Unnamed: 0'] not found in axis"